In [5]:
import requests
import os
import pandas as pd
from PIL import Image
from io import BytesIO
import time
import re

PEXELS_API = "pPHdVcUnYDnQ1KMmIwdEnxCpY5s5UQMzJtzbcZ7umkIlL7O2KRREbXJj"
PIXABAY_KEY = "55637926-af31d899d2daa8ad62517e2f0"
UNSPLASH_ACCESS_KEY = "yyxuw08ekQ4Sn5xwcY4KY9Xqj-I62Z1TyOJGxcPQAEA"


SAVE_DIR = "dataset_images"
os.makedirs(SAVE_DIR, exist_ok=True)

TARGET_IMAGES = 10000

PEXELS_HEADERS = {"Authorization": PEXELS_API}
UNSPLASH_HEADERS = {"Authorization": f"Client-ID {UNSPLASH_ACCESS_KEY}"}


def safe_name(text):
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


def download_image(url, path):
    try:
        r = requests.get(url, timeout=15)

        if r.status_code != 200:
            return False

        img = Image.open(BytesIO(r.content)).convert("RGB")

        if img.size[0] < 200 or img.size[1] < 200:
            return False

        img.save(path, quality=90)
        return True

    except Exception:
        return False


def scrape_pexels(query, per_page=20, pages=8):
    url = "https://api.pexels.com/v1/search"
    data = []
    qname = safe_name(query)

    for page in range(1, pages + 1):
        params = {
            "query": query,
            "per_page": min(per_page, 80),
            "page": page
        }

        r = requests.get(url, headers=PEXELS_HEADERS, params=params)

        if r.status_code == 429:
            print("Pexels rate limit reached. Stopping Pexels for:", query)
            break

        if r.status_code != 200:
            print("Pexels failed:", query, r.status_code, r.text[:200])
            break

        photos = r.json().get("photos", [])

        if not photos:
            break

        for i, photo in enumerate(photos):
            img_url = photo["src"]["large"]
            caption = photo.get("alt", "")

            if not caption or len(caption.split()) < 3:
                continue

            path = f"{SAVE_DIR}/pexels_{qname}_{page}_{i}.jpg"

            if download_image(img_url, path):
                data.append({
                    "image_path": path,
                    "caption": caption.lower(),
                    "source": "pexels",
                    "topic": query,
                    "image_url": img_url
                })

            time.sleep(0.2)

        time.sleep(1.5)

    print(f"Pexels {query} → {len(data)}")
    return data


def scrape_pixabay(query, per_page=20, pages=7):
    url = "https://pixabay.com/api/"
    data = []
    qname = safe_name(query)

    for page in range(1, pages + 1):
        params = {
            "key": PIXABAY_KEY,
            "q": query,
            "image_type": "photo",
            "safesearch": "true",
            "per_page": min(per_page, 100),
            "page": page,
            "min_width": 200,
            "min_height": 200
        }

        r = requests.get(url, params=params)

        if r.status_code == 429:
            print("Pixabay rate limit reached. Stopping Pixabay for:", query)
            break

        if r.status_code != 200:
            print("Pixabay failed:", query, r.status_code, r.text[:200])
            break

        hits = r.json().get("hits", [])

        if not hits:
            break

        for i, item in enumerate(hits):
            img_url = item.get("largeImageURL") or item.get("webformatURL")
            caption = item.get("tags", "")

            if not img_url or not caption:
                continue

            caption = caption.replace(",", " ").lower()

            if len(caption.split()) < 3:
                continue

            path = f"{SAVE_DIR}/pixabay_{qname}_{page}_{i}.jpg"

            if download_image(img_url, path):
                data.append({
                    "image_path": path,
                    "caption": caption,
                    "source": "pixabay",
                    "topic": query,
                    "image_url": img_url
                })

            time.sleep(0.2)

        time.sleep(1)

    print(f"Pixabay {query} → {len(data)}")
    return data


def scrape_unsplash(query, per_page=30, pages=1):
    url = "https://api.unsplash.com/search/photos"
    data = []
    qname = safe_name(query)

    for page in range(1, pages + 1):
        params = {
            "query": query,
            "per_page": min(per_page, 30),
            "page": page
        }

        r = requests.get(url, headers=UNSPLASH_HEADERS, params=params)

        if r.status_code == 403:
            print("Unsplash rate limit reached. Stopping Unsplash.")
            break

        if r.status_code != 200:
            print("Unsplash failed:", query, r.status_code, r.text[:200])
            break

        results = r.json().get("results", [])

        if not results:
            break

        for i, photo in enumerate(results):
            img_url = photo["urls"]["regular"]

            caption = (
                photo.get("alt_description")
                or photo.get("description")
                or query
            )

            if not caption or len(caption.split()) < 3:
                continue

            path = f"{SAVE_DIR}/unsplash_{qname}_{page}_{i}.jpg"

            if download_image(img_url, path):
                data.append({
                    "image_path": path,
                    "caption": caption.lower(),
                    "source": "unsplash",
                    "topic": query,
                    "image_url": img_url
                })

            time.sleep(0.2)

        time.sleep(1)

    print(f"Unsplash {query} → {len(data)}")
    return data


topics = [
    "person", "bicycle", "car", "motorcycle", "airplane",
    "bus", "train", "truck", "boat", "traffic light",
    "stop sign", "bench",

    "bird", "cat", "dog", "horse", "sheep", "cow",
    "elephant", "bear", "zebra",

    "people", "snowboard", "sports ball",
    "cup", "chair",

    "tv", "laptop", "remote",
    "book", "clock"
]


all_data = []

for topic in topics:
    print("\n==============================")
    print("TOPIC:", topic)
    print("==============================")

    all_data.extend(scrape_pexels(topic, per_page=20, pages=8))
    time.sleep(5)

    all_data.extend(scrape_pixabay(topic, per_page=20, pages=7))
    time.sleep(3)

    all_data.extend(scrape_unsplash(topic, per_page=30, pages=1))
    time.sleep(3)

    temp_df = pd.DataFrame(all_data)

    if len(temp_df) > 0:
        temp_df = temp_df.drop_duplicates(subset=["image_url"]).reset_index(drop=True)
        temp_df.to_csv("progress_dataset.csv", index=False)

        print("Current total:", len(temp_df))

        if len(temp_df) >= TARGET_IMAGES:
            print("Target reached!")
            break


df = pd.DataFrame(all_data)

if len(df) > 0:
    df = df.drop_duplicates(subset=["image_url"]).reset_index(drop=True)
    df.to_csv("final_combined_dataset.csv", index=False)

    print("\nFINAL STATS")
    print("TOTAL:", len(df))
    print(df["source"].value_counts())
    print(df["topic"].value_counts())
else:
    print("No images downloaded.")



TOPIC: person
Pexels person → 160
Pixabay person → 140
Unsplash person → 30
Current total: 308

TOPIC: bicycle
Pexels bicycle → 160
Pixabay bicycle → 140
Unsplash bicycle → 29
Current total: 630

TOPIC: car
Pexels car → 160
Pixabay car → 140
Unsplash car → 28
Current total: 939

TOPIC: motorcycle
Pexels motorcycle → 160
Pixabay motorcycle → 140
Unsplash motorcycle → 29
Current total: 1256

TOPIC: airplane
Pexels airplane → 160
Pixabay airplane → 140
Unsplash airplane → 28
Current total: 1568

TOPIC: bus
Pexels bus → 160
Pixabay bus → 140
Unsplash bus → 30
Current total: 1872

TOPIC: train
Pexels train → 160
Pixabay train → 140
Unsplash train → 29
Current total: 2184

TOPIC: truck
Pexels truck → 160
Pixabay truck → 139
Unsplash truck → 28
Current total: 2486

TOPIC: boat
Pexels boat → 160
Pixabay boat → 140
Unsplash boat → 29
Current total: 2790

TOPIC: traffic light
Pexels traffic light → 160
Pixabay traffic light → 140
Unsplash traffic light → 27
Current total: 3093

TOPIC: stop sign

In [7]:
import pandas as pd
import re
df=pd.read_csv("final_combined_dataset.csv")
df.head()

,image_path,caption,source,topic,image_url
0,dataset_images/pexels_person_1_0.jpg,grayscale portrait of a confident young man wi...,pexels,person,https://images.pexels.com/photos/10078860/pexe...
1,dataset_images/pexels_person_1_1.jpg,young woman in red blouse smiling indoors with...,pexels,person,https://images.pexels.com/photos/36794535/pexe...
2,dataset_images/pexels_person_1_2.jpg,close-up portrait of a young man wearing eyegl...,pexels,person,https://images.pexels.com/photos/2117283/pexel...
3,dataset_images/pexels_person_1_3.jpg,close-up portrait of a mature asian man with a...,pexels,person,https://images.pexels.com/photos/14004620/pexe...
4,dataset_images/pexels_person_1_4.jpg,indian man striking a pose with arm raised in ...,pexels,person,https://images.pexels.com/photos/11463528/pexe...


In [8]:
df.to_csv("1-preparation_DONE.csv")